# Running Granite Time Series IQR Ensemble R1 on GIFT-Eval

**This notebook reproduces the Granite Time Series IQR Ensemble R1 submission to the GIFT-Eval leaderboard.**

The ensemble combines four Granite time-series foundation models:

- [Granite PatchTST-FM R1](https://huggingface.co/ibm-granite/granite-timeseries-patchtst-fm-r1)
- [Granite PatchTST-FM R2](https://huggingface.co/ibm-granite/granite-timeseries-patchtst-fm-r2)
- [Granite FlowState R1](https://huggingface.co/ibm-granite/granite-timeseries-flowstate-r1), revision `r1.1`
- [Granite TTM R3](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r3)

The member forecasts are combined using IQR-weighted probabilistic aggregation. The published ensemble recipe is loaded from:

[the `recipes/granite-iqr` configuration in ibm-granite/granite-timeseries-ensemble-r1](https://huggingface.co/ibm-granite/granite-timeseries-ensemble-r1/tree/main/recipes/granite-iqr)

This notebook:

1. Installs Granite TSFM and GIFT-Eval.
2. Downloads the published ensemble configuration and member checkpoints from Hugging Face.
3. Loads the GIFT-Eval datasets and benchmark settings.
4. Runs a small smoke test.
5. Optionally runs the complete benchmark.
6. Validates the generated `all_results.csv` and member results.

## IQR-weighted aggregation

For each forecast position, the ensemble measures the uncertainty of each member using its interquartile range:

\[
\operatorname{IQR}_m = Q_m(0.75) - Q_m(0.25),
\]

where \(m\) identifies an ensemble member.

Members with narrower prediction intervals receive greater weight. The inverse-IQR scores are converted into normalized weights using a softmax:

\[
s_m = \frac{1}{\operatorname{IQR}_m + \epsilon},
\]

\[
w_m =
\frac{\exp(s_m / \tau)}
     {\sum_j \exp(s_j / \tau)},
\]

where:

- \(\epsilon\) provides numerical stability.
- \(\tau\) is the temperature controlling how concentrated the weights are.
- An optional maximum-weight setting can prevent one member from dominating the ensemble.

The weights vary across samples, forecast horizons, and target variables. They are applied to the members' quantile forecasts to produce the final probabilistic forecast.

A narrower interval indicates greater model confidence, but does not guarantee greater forecast accuracy. IQR weighting uses forecast sharpness as an aggregation signal.

## Installation

Run this notebook from within a GIFT-Eval checkout. The Granite checkout is reused if it already exists. Restart the kernel after changing an installed checkout or package version.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

gift_eval_root = Path.cwd().resolve().parent
notebook_dir = Path.cwd().resolve()
if not (gift_eval_root / "pyproject.toml").is_file() or not (gift_eval_root / "src/gift_eval").is_dir():
    raise RuntimeError("Run this notebook from the GIFT-Eval notebooks directory")

granite_source = notebook_dir / "granite-tsfm"
if not os.path.exists("granite-tsfm"):
    !git clone --branch granite_tsfm_ensemble --single-branch --depth 1 https://github.com/ibm-granite/granite-tsfm.git
else:
    print("Folder 'granite-tsfm' already exists. Skipping git clone.")

In [ ]:
%cd granite-tsfm
%pip install -q -e ".[notebooks]"
%cd ..
%pip install -q -e "..[baseline]"

sys.path.insert(0, str(granite_source.resolve()))
sys.path.insert(0, str((granite_source / "notebooks/hfdemo/gift_eval_ensemble").resolve()))

## Configuration sources

The replication uses two complementary configurations:

- The Hugging Face repository defines the ensemble recipe: member checkpoints, model revisions, quantile levels, aggregation method, and ensemble weights.
- The Granite TSFM experiment configuration defines GIFT-Eval execution details: datasets, aliases, forecast terms, seed, output names, device selection, and resume behavior.

The Hugging Face configuration is treated as the published model definition. The local experiment configuration contains the benchmark-specific settings required by the GIFT-Eval runner.

## Data and experiment configuration

Set `GIFT_EVAL_PATH` directly in the following cell, or leave it as `None` to use the `GIFT_EVAL` environment variable.

The path must point to the directory containing the downloaded GIFT-Eval data.

The ensemble configuration is downloaded from Hugging Face using `ProbabilisticEnsembleConfig.from_pretrained()`. The GIFT-Eval execution settings are loaded from the Granite TSFM checkout.

For the IQR ensemble, the Hugging Face configuration also defines:

- The IQR softmax temperature
- The optional maximum member weight
- Any prior member weights

These settings affect the generated forecasts and are part of the published model recipe rather than GIFT-Eval runtime configuration.

In [ ]:
import json
import os

from dotenv import load_dotenv
from tsfm_public.models.ensemble.configuration_ensemble import (
    ProbabilisticEnsembleConfig,
)

load_dotenv(gift_eval_root / ".env")

# GIFT-Eval data
GIFT_EVAL_PATH = None
if GIFT_EVAL_PATH is None:
    GIFT_EVAL_PATH = os.environ.get("GIFT_EVAL")

if not GIFT_EVAL_PATH or not Path(GIFT_EVAL_PATH).expanduser().is_dir():
    raise RuntimeError(
        "Set GIFT_EVAL_PATH or GIFT_EVAL to the downloaded dataset directory"
    )

os.environ["GIFT_EVAL"] = str(
    Path(GIFT_EVAL_PATH).expanduser().resolve()
)

# Ensemble recipe from Hugging Face
ENSEMBLE_REPO_ID = "ibm-granite/granite-timeseries-ensemble-r1"

ensemble_config = ProbabilisticEnsembleConfig.from_pretrained(
    ENSEMBLE_REPO_ID,
    subfolder="recipes/granite-iqr",
)

ensemble_config.validate()

# GIFT-Eval execution settings from the Granite TSFM checkout
experiment_config_path = (
    granite_source
    / "notebooks/hfdemo/gift_eval_ensemble/experiment_config.json"
)

if not experiment_config_path.is_file():
    raise FileNotFoundError(
        f"Missing GIFT-Eval experiment configuration: "
        f"{experiment_config_path}"
    )

experiment_config = json.loads(experiment_config_path.read_text())

from run_gift_eval import run_evaluation

In [ ]:
RUNNER_CONFIG_NAME = "iqr-weighted-ensemble-ibm-tsfm-granite-pt"
LEADERBOARD_MODEL_NAME = "Granite-timeseries-ensemble-r1-iqr"
gift_recipe = experiment_config["configurations"][RUNNER_CONFIG_NAME]

assert ensemble_config.aggregation_method == "iqr_weighted"
assert gift_recipe["ensemble"] == "iqr_weighted"
assert ensemble_config.quantile_levels == experiment_config["quantile_levels"]
assert len(ensemble_config.members) == len(gift_recipe["model_names"])

iqr_options = ensemble_config.iqr_weighted_options
assert gift_recipe["iqr_temperature"] == iqr_options["temperature"]
assert gift_recipe["iqr_max_weight"] == iqr_options["max_weight"]


In [ ]:
SETTINGS = experiment_config["defaults"].copy()
SETTINGS["model_name_config"] = RUNNER_CONFIG_NAME

SETTINGS.update(
    skip_processed=True,
    save_member_results=True,
)

output_root = Path(
    os.environ.get(
        "GIFT_EVAL_OUTPUT_ROOT",
        gift_eval_root / "outputs/granite_tsfm_iqr_ensemble",
    )
).expanduser().resolve()

print(f"Leaderboard model name: {LEADERBOARD_MODEL_NAME}")
print(f"Runner configuration: {RUNNER_CONFIG_NAME}")
print(f"Ensemble repository: {ENSEMBLE_REPO_ID}")
print(f"GIFT-Eval data location: {os.environ['GIFT_EVAL']}")
print(f"Output root: {output_root}")
print(ensemble_config)


## Environment record

In [ ]:
import platform

import numpy as np
import pandas as pd
import torch
import transformers

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("Granite revision:", subprocess.run(["git", "-C", str(granite_source), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("GIFT-Eval revision:", subprocess.run(["git", "-C", str(gift_eval_root), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())

## Smoke test

Run the smoke test before starting the complete benchmark.

The smoke test evaluates `us_births/M` and verifies:

- GIFT-Eval data access
- Hugging Face model downloads
- Loading of all four ensemble members
- CUDA or CPU inference
- Generation of member quantile forecasts
- Calculation of IQR-derived member weights
- IQR-weighted aggregation
- Metric and result generation

The smoke test writes to `outputs/granite_tsfm_iqr_ensemble/smoke` and does not modify the submitted leaderboard result.

In [ ]:
smoke_settings = {
    **SETTINGS,
    "datasets": ["us_births/M"],
    "out_dir": output_root / "smoke",
}
smoke_csv = run_evaluation(**smoke_settings)
display(pd.read_csv(smoke_csv).sort_values("dataset"))

## Full benchmark

Set `RUN_FULL_BENCHMARK=True` after the smoke test succeeds, or set the environment variable `RUN_FULL_BENCHMARK=true` for a batch run. The run is resumable and writes beneath the ignored `outputs/` directory, leaving the committed leaderboard result unchanged.

In [ ]:
RUN_FULL_BENCHMARK = os.environ.get("RUN_FULL_BENCHMARK", "false").lower() in {"1", "true", "yes"}
full_csv = output_root / "full" / RUNNER_CONFIG_NAME / SETTINGS["out_name"]
if RUN_FULL_BENCHMARK:
    full_settings = {
        **SETTINGS,
        "datasets": None,
        "out_dir": output_root / "full",
    }
    full_csv = run_evaluation(**full_settings)
else:
    print("Full evaluation not started.")


## Validate and compare results

After the full benchmark completes, the following checks are applied:

1. Every expected GIFT-Eval task is present exactly once.
2. The output follows the required leaderboard schema.
3. All evaluation metrics are finite.
4. The leaderboard model name is consistent across rows.
5. A complete result file exists for every ensemble member.
6. When a submitted result is available locally, the regenerated metrics are compared with it using a small numerical tolerance.

Minor floating-point differences can occur between compatible hardware and software environments. Larger or systematic differences should be investigated using the saved member results and execution log.

If the ensemble result differs while all saved member results match, inspect the IQR aggregation settings and implementation first. In particular, verify the temperature, maximum-weight setting, quantile interpolation, and member ordering.

In [ ]:
if full_csv.is_file():
    results = pd.read_csv(full_csv).sort_values("dataset").reset_index(drop=True)
    expected_rows = (
        len(experiment_config["datasets"]["short"])
        + 2 * len(experiment_config["datasets"]["medium_long"])
    )
    metrics = results.filter(regex=r"^eval_metrics/")
    assert len(results) == results["dataset"].nunique() == expected_rows
    assert results.shape[1] == 15
    assert results["model"].eq(RUNNER_CONFIG_NAME).all()
    assert metrics.shape[1] == 11 and np.isfinite(metrics.to_numpy()).all()

    expected_members = experiment_config["configurations"][RUNNER_CONFIG_NAME]["model_names"]

    for member_name in expected_members:
        member_csv = (
            full_csv.parent
            / "members"
            / member_name
            / SETTINGS["out_name"]
        )
        assert member_csv.is_file(), f"Missing member results: {member_name}"

        member_results = pd.read_csv(member_csv)
        assert not member_results.empty, (
            f"No member results produced for {member_name}"
        )
        assert member_results["dataset"].is_unique, (
            f"Duplicate member results found for {member_name}"
        )

    submitted_csv = gift_eval_root / "results" / LEADERBOARD_MODEL_NAME / "all_results.csv"
    if submitted_csv.is_file():
        submitted = pd.read_csv(submitted_csv).sort_values("dataset").reset_index(drop=True)
        generated_for_comparison = results.copy()
        generated_for_comparison["model"] = LEADERBOARD_MODEL_NAME
        pd.testing.assert_frame_equal(
            generated_for_comparison,
            submitted,
            check_dtype=False,
            rtol=1e-5,
            atol=1e-7,
        )
        print(f"Generated metrics match {submitted_csv}")
    else:
        print(f"Copy the validated CSV to {submitted_csv} for submission.")
    display(results)
else:
    print("Run the full benchmark before validating results.")